In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery, Filter
from weaviate.classes.tenants import Tenant

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "TenantDocument"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

tenant_documents = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    multi_tenancy_config=wvc.config.Configure.multi_tenancy(
        enabled=True,
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="department",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: TenantDocument


In [5]:
tenant_documents.tenants.create(
    tenants=[
        Tenant(name="company_ai"),
        Tenant(name="company_dev"),
        Tenant(name="company_office"),
    ]
)

print("Tenants created.")

Tenants created.


In [6]:
tenants = tenant_documents.tenants.get()

for tenant_name, tenant_info in tenants.items():
    print(tenant_name, tenant_info)

company_office name='company_office' activityStatusInternal=<TenantActivityStatus.ACTIVE: 'ACTIVE'> activityStatus=<_TenantActivistatusServerValues.HOT: 'HOT'>
company_ai name='company_ai' activityStatusInternal=<TenantActivityStatus.ACTIVE: 'ACTIVE'> activityStatus=<_TenantActivistatusServerValues.HOT: 'HOT'>
company_dev name='company_dev' activityStatusInternal=<TenantActivityStatus.ACTIVE: 'ACTIVE'> activityStatus=<_TenantActivistatusServerValues.HOT: 'HOT'>


In [7]:
tenant_data = {
    "company_ai": [
        {
            "title": "GPU Workstation Guide",
            "content": "A GPU workstation should have a powerful NVIDIA GPU, enough VRAM, fast NVMe storage and sufficient RAM for local AI experiments.",
            "category": "Hardware",
            "department": "AI Lab",
        },
        {
            "title": "Local LLM Deployment",
            "content": "Local LLM deployment requires model files, quantization strategy, GPU acceleration and enough memory for inference workloads.",
            "category": "AI",
            "department": "AI Lab",
        },
        {
            "title": "Vector Database Usage",
            "content": "Vector databases store embeddings and enable semantic search, hybrid search and retrieval augmented generation.",
            "category": "Vector Search",
            "department": "AI Lab",
        },
    ],
    "company_dev": [
        {
            "title": "Backend Development Environment",
            "content": "A Python backend environment usually includes FastAPI, pytest, Ruff, uv, Docker and CI pipelines.",
            "category": "Backend",
            "department": "Engineering",
        },
        {
            "title": "Git Workflow",
            "content": "A clean Git workflow uses feature branches, pull requests, conventional commits and automated checks before merging.",
            "category": "Git",
            "department": "Engineering",
        },
        {
            "title": "Testing Strategy",
            "content": "A good testing strategy includes unit tests, integration tests, fixtures, parametrization and coverage reports.",
            "category": "Testing",
            "department": "Engineering",
        },
    ],
    "company_office": [
        {
            "title": "Office Hardware Policy",
            "content": "Office laptops should be lightweight, reliable, encrypted and suitable for email, documents and video meetings.",
            "category": "Office IT",
            "department": "Administration",
        },
        {
            "title": "Meeting Room Equipment",
            "content": "Meeting rooms should include a webcam, microphone, speakers, display screen and stable network connection.",
            "category": "Office IT",
            "department": "Administration",
        },
        {
            "title": "Printer Maintenance",
            "content": "Printers require regular toner replacement, paper restocking, driver updates and network configuration checks.",
            "category": "Office Operations",
            "department": "Administration",
        },
    ],
}

In [8]:
inserted_uuids = {}

for tenant_name, objects in tenant_data.items():
    tenant_collection = tenant_documents.with_tenant(tenant_name)
    inserted_uuids[tenant_name] = []

    for obj in objects:
        uuid = tenant_collection.data.insert(obj)
        inserted_uuids[tenant_name].append(uuid)

        print(f"{tenant_name}: {obj['title']} -> {uuid}")

company_ai: GPU Workstation Guide -> f536238f-9528-452f-b57c-5d5a2bca8924
company_ai: Local LLM Deployment -> 749d36b5-6d30-428f-b66d-587a2e5ff7bf
company_ai: Vector Database Usage -> b8a9ac26-881e-4a4e-bb19-1f1186b75e9a
company_dev: Backend Development Environment -> 3dc68828-ba2c-4de7-8890-75cd19f2e813
company_dev: Git Workflow -> 3285e68c-3d9a-4a32-9078-3d0960021df0
company_dev: Testing Strategy -> c755d20b-3d01-4bc8-b5dc-141de183ec79
company_office: Office Hardware Policy -> a0d544a7-3f3b-4197-98e1-455fea8fc7df
company_office: Meeting Room Equipment -> 8443186d-a376-4a65-9ba6-f31b5dbbfa9a
company_office: Printer Maintenance -> 88283ebd-28fd-4d2b-9e2e-231cdcbd5ac4


In [9]:
for tenant_name in tenant_data:
    tenant_collection = tenant_documents.with_tenant(tenant_name)

    count = tenant_collection.aggregate.over_all(total_count=True)

    print(tenant_name, "objects:", count.total_count)

company_ai objects: 3
company_dev objects: 3
company_office objects: 3


In [10]:
company_ai_docs = tenant_documents.with_tenant("company_ai")

response = company_ai_docs.query.fetch_objects(
    limit=10,
    return_properties=["title", "category", "department", "content"],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Category:", obj.properties["category"])
    print("Department:", obj.properties["department"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Title: Local LLM Deployment
Category: AI
Department: AI Lab
Content: Local LLM deployment requires model files, quantization strategy, GPU acceleration and enough memory for inference workloads.
--------------------------------------------------------------------------------
Title: Vector Database Usage
Category: Vector Search
Department: AI Lab
Content: Vector databases store embeddings and enable semantic search, hybrid search and retrieval augmented generation.
--------------------------------------------------------------------------------
Title: GPU Workstation Guide
Category: Hardware
Department: AI Lab
Content: A GPU workstation should have a powerful NVIDIA GPU, enough VRAM, fast NVMe storage and sufficient RAM for local AI experiments.
--------------------------------------------------------------------------------


In [11]:
query = "hardware for AI models and GPU workloads"

for tenant_name in ["company_ai", "company_dev", "company_office"]:
    print("\nTENANT:", tenant_name)

    tenant_collection = tenant_documents.with_tenant(tenant_name)

    response = tenant_collection.query.near_text(
        query=query,
        limit=3,
        return_properties=["title", "category", "content"],
        return_metadata=MetadataQuery(distance=True),
    )

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Category:", obj.properties["category"])
        print("Content:", obj.properties["content"])
        print("-" * 80)


TENANT: company_ai
Distance: 0.29012471437454224
Title: GPU Workstation Guide
Category: Hardware
Content: A GPU workstation should have a powerful NVIDIA GPU, enough VRAM, fast NVMe storage and sufficient RAM for local AI experiments.
--------------------------------------------------------------------------------
Distance: 0.4659334421157837
Title: Local LLM Deployment
Category: AI
Content: Local LLM deployment requires model files, quantization strategy, GPU acceleration and enough memory for inference workloads.
--------------------------------------------------------------------------------
Distance: 0.6759874820709229
Title: Vector Database Usage
Category: Vector Search
Content: Vector databases store embeddings and enable semantic search, hybrid search and retrieval augmented generation.
--------------------------------------------------------------------------------

TENANT: company_dev
Distance: 0.7299228310585022
Title: Backend Development Environment
Category: Backend
Conten

In [12]:
query = "testing Python backend applications"

for tenant_name in ["company_ai", "company_dev", "company_office"]:
    print("\nTENANT:", tenant_name)

    tenant_collection = tenant_documents.with_tenant(tenant_name)

    response = tenant_collection.query.near_text(
        query=query,
        limit=3,
        return_properties=["title", "category", "content"],
        return_metadata=MetadataQuery(distance=True),
    )

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Category:", obj.properties["category"])
        print("Content:", obj.properties["content"])
        print("-" * 80)


TENANT: company_ai
Distance: 0.8193438649177551
Title: GPU Workstation Guide
Category: Hardware
Content: A GPU workstation should have a powerful NVIDIA GPU, enough VRAM, fast NVMe storage and sufficient RAM for local AI experiments.
--------------------------------------------------------------------------------
Distance: 0.8304173946380615
Title: Local LLM Deployment
Category: AI
Content: Local LLM deployment requires model files, quantization strategy, GPU acceleration and enough memory for inference workloads.
--------------------------------------------------------------------------------
Distance: 0.8959860801696777
Title: Vector Database Usage
Category: Vector Search
Content: Vector databases store embeddings and enable semantic search, hybrid search and retrieval augmented generation.
--------------------------------------------------------------------------------

TENANT: company_dev
Distance: 0.44004273414611816
Title: Backend Development Environment
Category: Backend
Conten

In [13]:
query = "testing Python backend applications"

for tenant_name in ["company_ai", "company_dev", "company_office"]:
    print("\nTENANT:", tenant_name)

    tenant_collection = tenant_documents.with_tenant(tenant_name)

    response = tenant_collection.query.near_text(
        query=query,
        limit=3,
        return_properties=["title", "category", "content"],
        return_metadata=MetadataQuery(distance=True),
    )

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Category:", obj.properties["category"])
        print("Content:", obj.properties["content"])
        print("-" * 80)


TENANT: company_ai
Distance: 0.8193438649177551
Title: GPU Workstation Guide
Category: Hardware
Content: A GPU workstation should have a powerful NVIDIA GPU, enough VRAM, fast NVMe storage and sufficient RAM for local AI experiments.
--------------------------------------------------------------------------------
Distance: 0.8304173946380615
Title: Local LLM Deployment
Category: AI
Content: Local LLM deployment requires model files, quantization strategy, GPU acceleration and enough memory for inference workloads.
--------------------------------------------------------------------------------
Distance: 0.8959860801696777
Title: Vector Database Usage
Category: Vector Search
Content: Vector databases store embeddings and enable semantic search, hybrid search and retrieval augmented generation.
--------------------------------------------------------------------------------

TENANT: company_dev
Distance: 0.44004273414611816
Title: Backend Development Environment
Category: Backend
Conten

In [14]:
tenant_collection = tenant_documents.with_tenant("company_dev")

response = tenant_collection.query.hybrid(
    query="pytest coverage automated checks",
    alpha=0.5,
    limit=3,
    return_properties=["title", "category", "content"],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Score:", obj.metadata.score)
    print("Title:", obj.properties["title"])
    print("Category:", obj.properties["category"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Score: 0.6356155872344971
Title: Git Workflow
Category: Git
Content: A clean Git workflow uses feature branches, pull requests, conventional commits and automated checks before merging.
--------------------------------------------------------------------------------
Score: 0.5
Title: Testing Strategy
Category: Testing
Content: A good testing strategy includes unit tests, integration tests, fixtures, parametrization and coverage reports.
--------------------------------------------------------------------------------
Score: 0.0
Title: Backend Development Environment
Category: Backend
Content: A Python backend environment usually includes FastAPI, pytest, Ruff, uv, Docker and CI pipelines.
--------------------------------------------------------------------------------


In [15]:
tenant_collection = tenant_documents.with_tenant("company_ai")

response = tenant_collection.query.near_text(
    query="semantic search and embeddings",
    filters=Filter.by_property("category").equal("Vector Search"),
    limit=3,
    return_properties=["title", "category", "department", "content"],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Category:", obj.properties["category"])
    print("Department:", obj.properties["department"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.4649873971939087
Title: Vector Database Usage
Category: Vector Search
Department: AI Lab
Content: Vector databases store embeddings and enable semantic search, hybrid search and retrieval augmented generation.
--------------------------------------------------------------------------------


In [16]:
tenant_collection = tenant_documents.with_tenant("company_ai")

response = tenant_collection.generate.near_text(
    query="local AI workstation",
    grouped_task=(
        "Explain what this tenant's documents say about local AI workstation requirements. "
        "Use only the retrieved tenant-specific context."
    ),
    limit=3,
    return_properties=["title", "category", "content"],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print("-", obj.properties["title"], "|", obj.properties["category"])

The tenant's documents outline specific requirements for local AI workstations. A GPU workstation should be equipped with a powerful NVIDIA GPU, sufficient VRAM, fast NVMe storage, and ample RAM to support local AI experiments. For local deployment of large language models (LLMs), the system must accommodate model files, have a quantization strategy in place, utilize GPU acceleration, and possess enough memory to handle inference workloads. Additionally, vector databases are essential for storing embeddings and facilitating semantic search, hybrid search, and retrieval-augmented generation.

Sources:
- GPU Workstation Guide | Hardware
- Local LLM Deployment | AI
- Vector Database Usage | Vector Search


In [17]:
def ask_tenant(tenant_name: str, question: str, *, limit: int = 3) -> None:
    tenant_collection = tenant_documents.with_tenant(tenant_name)

    response = tenant_collection.generate.near_text(
        query=question,
        grouped_task=(
            "Answer the question using only this tenant's retrieved documents. "
            "Keep the answer concise and practical. "
            "If the retrieved tenant context is insufficient, say so clearly."
        ),
        limit=limit,
        return_properties=["title", "category", "content"],
    )

    print("TENANT:", tenant_name)
    print("QUESTION:", question)
    print("\nANSWER:")
    print(response.generated)

    print("\nSOURCES:")
    for obj in response.objects:
        print("-", obj.properties["title"], "|", obj.properties["category"])

In [18]:
ask_tenant(
    "company_ai",
    "What hardware is recommended for local AI experiments?",
)

TENANT: company_ai
QUESTION: What hardware is recommended for local AI experiments?

ANSWER:
The retrieved documents provide guidance on setting up a GPU workstation as well as information on local LLM deployment and vector database usage. However, if you have a specific question, please provide more details for a tailored response.

SOURCES:
- GPU Workstation Guide | Hardware
- Local LLM Deployment | AI
- Vector Database Usage | Vector Search


In [19]:
ask_tenant(
    "company_dev",
    "What does the tenant know about Python backend testing?",
)

TENANT: company_dev
QUESTION: What does the tenant know about Python backend testing?

ANSWER:
The retrieved documents provide information on a Python backend environment, a good testing strategy, and a clean Git workflow. However, they do not contain specific context or answers to a particular question. Please provide more details or a specific question for a more targeted response.

SOURCES:
- Backend Development Environment | Backend
- Testing Strategy | Testing
- Git Workflow | Git


In [20]:
ask_tenant(
    "company_office",
    "What equipment is needed for meeting rooms?",
)

TENANT: company_office
QUESTION: What equipment is needed for meeting rooms?

ANSWER:
The retrieved documents do not provide information about how often maintenance is needed for office laptops. Therefore, I cannot answer the question regarding maintenance frequency for office laptops.

SOURCES:
- Meeting Room Equipment | Office IT
- Office Hardware Policy | Office IT
- Printer Maintenance | Office Operations


In [21]:
ask_tenant(
    "company_office",
    "What does the tenant know about GPU workstations for AI?",
)

TENANT: company_office
QUESTION: What does the tenant know about GPU workstations for AI?

ANSWER:
The retrieved documents do not provide specific information regarding the question posed. Please clarify or provide further context for a more targeted response.

SOURCES:
- Office Hardware Policy | Office IT
- Meeting Room Equipment | Office IT
- Printer Maintenance | Office Operations


In [22]:
tenants = tenant_documents.tenants.get()

for tenant_name in tenants.keys():
    print("\nTENANT:", tenant_name)

    tenant_collection = tenant_documents.with_tenant(tenant_name)

    for item in tenant_collection.iterator():
        print(item.properties)


TENANT: company_ai
{'department': 'AI Lab', 'category': 'AI', 'title': 'Local LLM Deployment', 'content': 'Local LLM deployment requires model files, quantization strategy, GPU acceleration and enough memory for inference workloads.'}
{'department': 'AI Lab', 'category': 'Vector Search', 'title': 'Vector Database Usage', 'content': 'Vector databases store embeddings and enable semantic search, hybrid search and retrieval augmented generation.'}
{'department': 'AI Lab', 'title': 'GPU Workstation Guide', 'category': 'Hardware', 'content': 'A GPU workstation should have a powerful NVIDIA GPU, enough VRAM, fast NVMe storage and sufficient RAM for local AI experiments.'}

TENANT: company_dev
{'department': 'Engineering', 'title': 'Git Workflow', 'category': 'Git', 'content': 'A clean Git workflow uses feature branches, pull requests, conventional commits and automated checks before merging.'}
{'department': 'Engineering', 'title': 'Backend Development Environment', 'category': 'Backend', '

In [23]:
client.close()